In [ ]:
%pip install fastparquet

# Constrained sOED Planning
This notebook is isolated from exploratory code and uses a dedicated constrained agent.

Hard constraints enforced during planning:
- Mass1 < 30
- Mass1 + Mass2 < 30
- Mass1 + Mass2 >= 3 (minimum load wall)
- Boost pressure within ±0.5 bar of: Boost = 0.0922 * (Mass1 + Mass2) + 0.8378
- Boost pressure >= 1.0 bar (ambient floor)
- Boost pressure <= 3.8 bar (TC roof)
- BR limit by load band:
  - 0 < Mass1 < 10: 0.5 < Mass2 < 3.5
  - 10 < Mass1 < 20: 0.9 < Mass2 < 3.0
  - 20 < Mass1 < 30: 0.0 < Mass2 < 1.5
- VVA limit by load band:
  - 0 < Mass1 < 10: IVO 350-435, IVC 500-540, EVO 128-218, EVC 270-350
  - 10 <= Mass1 < 20: IVO 330-390, IVC 500-570, EVO 128-218, EVC 330-370
  - 20 <= Mass1 < 30: IVO 345-365, IVC 495-535, EVO 128-218, EVC 345-355

The notebook keeps these as hard feasibility checks on candidate paths.

In [ ]:
%cd D:/shahnawaz/uva/main
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src.data_loader import DataLoader
from src.scaler import Scaler
from src.constant_manager import ConstantManager
from src.soed.agents.constrained_multistep_mimo_agent import ConstrainedMultiStepMIMOAgent

torch.set_default_dtype(torch.float64)
torch.manual_seed(42)

In [ ]:
PD_DATA_FILE = 'rcci_cleaned_data_v4_5.parquet'

cm = ConstantManager()
input_features = cm.RAW_REDUCED_INPUT_COLUMNS
output_features = cm.RAW_OUTPUT_COLUMNS

required = {'Boost pressure', 'Mass1', 'Mass2'}
missing = required.difference(set(input_features))
if missing:
    raise ValueError(f'Missing required constrained features in input_features: {missing}')

pd_df = DataLoader(file_path=PD_DATA_FILE).load_data()

scaler_x = Scaler()
scaler_y = Scaler()

scaled_df = pd_df[input_features + output_features].copy()
scaled_df[input_features] = scaler_x.fit_transform(scaled_df, input_features)
scaled_df[output_features] = scaler_y.fit_transform(scaled_df, output_features)

inputs_scaled = scaled_df[input_features].to_numpy()
outputs_scaled = scaled_df[output_features].to_numpy()

scaled_df[input_features + output_features].head()

In [ ]:
# Build bounds from unscaled engineering bounds, then map to scaled domain
unscaled_bounds = cm.UNSCALED_BOUNDS
ordered_bounds = [unscaled_bounds[f] for f in input_features]

# Keep feature names attached so StandardScaler does not warn about missing names.
unscaled_bounds_df = pd.DataFrame(
    np.array(ordered_bounds).T,
    columns=input_features,
)
scaled_bounds_df = pd.DataFrame(
    scaler_x.transform(unscaled_bounds_df),
    columns=input_features,
)
scaled_bounds = torch.tensor(scaled_bounds_df.to_numpy().T, dtype=torch.float64)

print("scaled_bounds shape:", tuple(scaled_bounds.shape))
scaled_bounds

In [ ]:
# Quick data sanity check before GP fit
x_arr = np.asarray(inputs_scaled, dtype=float)
y_arr = np.asarray(outputs_scaled, dtype=float)

x_bad = ~np.isfinite(x_arr)
y_bad = ~np.isfinite(y_arr)

print("Inputs shape:", x_arr.shape, "| Outputs shape:", y_arr.shape)
print("Input non-finite count:", int(x_bad.sum()))
print("Output non-finite count:", int(y_bad.sum()))

if x_bad.any():
    bad_cols_x = [input_features[i] for i in np.where(x_bad.any(axis=0))[0]]
    print("Input columns with NaN/Inf:", bad_cols_x)

if y_bad.any():
    bad_cols_y = [output_features[i] for i in np.where(y_bad.any(axis=0))[0]]
    print("Output columns with NaN/Inf:", bad_cols_y)

print("Rows fully finite:", int((np.isfinite(x_arr).all(axis=1) & np.isfinite(y_arr).all(axis=1)).sum()))

In [ ]:
from pathlib import Path
import gpytorch

MODEL_BUNDLE_PATH = Path("model_artifacts/constrained_agent_bundle_v1.pth")
REUSE_SAVED_MODELS = True

# Planner behavior knobs
USE_DISTANCE_PENALTY = True
DISTANCE_WEIGHT = 0.05  # used only when USE_DISTANCE_PENALTY=True
planner_w_dist = DISTANCE_WEIGHT if USE_DISTANCE_PENALTY else 0.0

# Additional freedom knobs
INCLUDE_LOCAL_PATHS = True
LOCAL_PATH_FRACTION = 0.10  # local candidates as fraction of Sobol count
FEASIBLE_MARGIN_WEIGHT = 0.0

# Feasibility retry knobs
REQUIRE_FULLY_FEASIBLE_PATH = True
MAX_PLAN_RETRIES = 3
BASE_NUM_SCENARIOS = 2048
SCENARIO_GROWTH_PER_RETRY = 0.30  # grows candidate count each retry

agent = ConstrainedMultiStepMIMOAgent(
    bounds=scaled_bounds,
    feature_names=input_features,
    scaler_x=scaler_x,
    mass1_name='Mass1',
    mass2_name='Mass2',
    boost_name='Boost pressure',
    load_limit=30.0,
    boost_slope=0.0922,
    boost_intercept=0.8378,
    boost_band=0.5,
    min_load=3.0,
    ambient_pressure=1.0,
    tc_boost_limit=3.8,
 )

if REUSE_SAVED_MODELS and MODEL_BUNDLE_PATH.exists():
    loaded = agent.load_bundle(MODEL_BUNDLE_PATH)
    print(f"Loaded constrained surrogate bundle from: {MODEL_BUNDLE_PATH}")
    print(f"Bundle metadata keys: {list(loaded.keys())}")
else:
    agent.fit_data(inputs_scaled, outputs_scaled)
    saved_path = agent.save_bundle(
        MODEL_BUNDLE_PATH,
        extra_metadata={
            "source_notebook": "soed_constrained_planning.ipynb",
            "input_features": input_features,
            "output_features": output_features,
        },
    )
    print(f"Trained and saved constrained surrogate bundle to: {saved_path}")

print(f"Distance penalty enabled: {USE_DISTANCE_PENALTY} | w_dist={planner_w_dist}")
print(f"Include local RW paths: {INCLUDE_LOCAL_PATHS} | local_path_fraction={LOCAL_PATH_FRACTION}")
print(f"Feasible interior margin weight: {FEASIBLE_MARGIN_WEIGHT}")
print(f"Require fully feasible path: {REQUIRE_FULLY_FEASIBLE_PATH} | max retries: {MAX_PLAN_RETRIES}")

def _path_is_hard_feasible(a, p):
    with torch.no_grad():
        feasible_mask = a._feasible_mask(p.unsqueeze(0))
    return bool(feasible_mask.reshape(-1)[0].item())

def _path_objective(a, p, curr, w_dist, feasible_margin_weight):
    p_batch = p.unsqueeze(0)
    with torch.no_grad(), gpytorch.settings.cholesky_jitter(1e-4), gpytorch.settings.fast_pred_var(False):
        total_ig = torch.zeros(1, dtype=torch.float64, device=p.device)
        for model, lik in zip(a.models, a.likelihoods):
            cov = model.posterior(p_batch).distribution.covariance_matrix
            if cov.ndim == 4:
                cov = cov.squeeze(0)
            if not torch.isfinite(cov).all():
                return -float('inf')
            m = torch.eye(p.shape[0], dtype=cov.dtype, device=cov.device) + cov / lik.noise.clamp_min(1e-8)
            try:
                ig = torch.linalg.cholesky(m).diagonal(dim1=-2, dim2=-1).log().sum(dim=-1)
            except RuntimeError:
                ig = 0.5 * torch.linalg.slogdet(m)[1]
            total_ig += torch.where(torch.isfinite(ig), ig, torch.full_like(ig, -1e6))

        ls_eff = torch.min(
            torch.stack([m.covar_module.base_kernel.lengthscale.squeeze().detach() for m in a.models]),
            dim=0,
        ).values
        d0 = torch.sqrt((((p[0] - curr) / ls_eff) ** 2).sum())
        dstep = torch.sqrt((((p[1:] - p[:-1]) / ls_eff) ** 2).sum(dim=-1)).sum() if p.shape[0] > 1 else 0.0
        score = total_ig.squeeze() - w_dist * (d0 + dstep)

        feasible = _path_is_hard_feasible(a, p)
        if feasible:
            score = score - float(feasible_margin_weight) * a._interior_margin_penalty(p_batch).squeeze()
        else:
            score = score - 50.0 * a._constraint_penalty(p_batch).squeeze() - 10.0 * a._interior_margin_penalty(p_batch).squeeze()
        return float(score.item())

current_loc = agent.X[-1:]
curr = current_loc.squeeze()
attempt_records = []

for attempt in range(1, MAX_PLAN_RETRIES + 1):
    num_scenarios = int(round(BASE_NUM_SCENARIOS * (1.0 + SCENARIO_GROWTH_PER_RETRY * (attempt - 1))))
    candidate = agent.plan_multistep_batch(
        current_location=current_loc,
        q_steps=3,
        num_scenarios=num_scenarios,
        w_dist=planner_w_dist,
        enforce_feasible_sampling=True,
        enforce_feasible_sobol=True,
        include_local_paths=INCLUDE_LOCAL_PATHS,
        local_path_fraction=LOCAL_PATH_FRACTION,
        feasible_margin_weight=FEASIBLE_MARGIN_WEIGHT,
    )

    is_feasible = _path_is_hard_feasible(agent, candidate)
    score = _path_objective(
        agent,
        candidate,
        curr=curr,
        w_dist=planner_w_dist,
        feasible_margin_weight=FEASIBLE_MARGIN_WEIGHT,
    )

    attempt_records.append({
        'attempt': attempt,
        'num_scenarios': num_scenarios,
        'path': candidate.detach().clone(),
        'hard_feasible': is_feasible,
        'objective_score': score,
    })
    print(f"Attempt {attempt:02d}: scenarios={num_scenarios} | hard-feasible={is_feasible} | score={score:.4f}")

if REQUIRE_FULLY_FEASIBLE_PATH:
    feasible_attempts = [r for r in attempt_records if r['hard_feasible']]
    if feasible_attempts:
        best = max(feasible_attempts, key=lambda r: r['objective_score'])
        path_scaled = best['path']
        print(
            f"Selected best feasible attempt {best['attempt']} "
            f"(scenarios={best['num_scenarios']}, score={best['objective_score']:.4f})"
        )
    else:
        best = max(attempt_records, key=lambda r: r['objective_score'])
        path_scaled = best['path']
        print("Warning: retries exhausted; returning best available soft-constrained path.")
        print(
            f"Selected fallback attempt {best['attempt']} "
            f"(scenarios={best['num_scenarios']}, score={best['objective_score']:.4f})"
        )
else:
    best = max(attempt_records, key=lambda r: r['objective_score'])
    path_scaled = best['path']
    print(
        f"Selected best overall attempt {best['attempt']} "
        f"(scenarios={best['num_scenarios']}, score={best['objective_score']:.4f}, "
        f"hard-feasible={best['hard_feasible']})"
    )

path_feasible = _path_is_hard_feasible(agent, path_scaled)
last_attempt = int(best['attempt'])

path_scaled

In [ ]:
# Convert recommendation to physical units and verify all constraints
path_original = []
for p in path_scaled:
    row = []
    for i in range(len(input_features)):
        row.append(scaler_x.inverse_transform(p[i].item(), i))
    path_original.append(row)

path_original = pd.DataFrame(path_original, columns=input_features)

mass_sum = path_original['Mass1'] + path_original['Mass2']
boost_center = 0.0922 * mass_sum + 0.8378

# Pull hard bounds from agent so validation mirrors planner logic.
min_load = float(getattr(agent, 'min_load', 3.0))
max_load = float(getattr(agent, 'load_limit', 30.0))
ambient_pressure = float(getattr(agent, 'ambient_pressure', 1.0))
tc_boost_limit = float(getattr(agent, 'tc_boost_limit', 3.8))

path_original['Mass1+Mass2'] = mass_sum
path_original['Boost_low_raw'] = boost_center - 0.5
path_original['Boost_high_raw'] = boost_center + 0.5
path_original['Boost_low'] = np.maximum(path_original['Boost_low_raw'], ambient_pressure)
path_original['Boost_high'] = np.minimum(path_original['Boost_high_raw'], tc_boost_limit)

# Load check aligned with planner constraints.
path_original['Load_ok'] = (
    (path_original['Mass1'] >= 0.0) &
    (path_original['Mass1'] < max_load) &
    (mass_sum >= min_load) &
    (mass_sum < max_load)
)

# Boost check uses slanted band clipped by floor/roof, aligned with planner.
path_original['Boost_ok'] = (
    (path_original['Boost pressure'] >= path_original['Boost_low']) &
    (path_original['Boost pressure'] <= path_original['Boost_high'])
)

# BR limit by Mass1 band (aligned with planner band edges).
path_original['BR_ok'] = (
    ((path_original['Mass1'] >= 0.0) & (path_original['Mass1'] < 10.0) & (path_original['Mass2'] > 0.5) & (path_original['Mass2'] < 3.5)) |
    ((path_original['Mass1'] >= 10.0) & (path_original['Mass1'] < 20.0) & (path_original['Mass2'] > 0.9) & (path_original['Mass2'] < 3.0)) |
    ((path_original['Mass1'] >= 20.0) & (path_original['Mass1'] < 30.0) & (path_original['Mass2'] > 0.0) & (path_original['Mass2'] < 1.5))
)

# VVA limit by Mass1 band
low_load = (path_original['Mass1'] >= 0.0) & (path_original['Mass1'] < 10.0)
mid_load = (path_original['Mass1'] >= 10.0) & (path_original['Mass1'] < 20.0)
high_load = (path_original['Mass1'] >= 20.0) & (path_original['Mass1'] < 30.0)

path_original['VVA_ok'] = (
    (
        low_load &
        (path_original['IVO'] >= 350.0) & (path_original['IVO'] <= 435.0) &
        (path_original['IVC'] >= 500.0) & (path_original['IVC'] <= 540.0) &
        (path_original['EVO'] >= 128.0) & (path_original['EVO'] <= 218.0) &
        (path_original['EVC'] >= 270.0) & (path_original['EVC'] <= 350.0)
    ) |
    (
        mid_load &
        (path_original['IVO'] >= 330.0) & (path_original['IVO'] <= 390.0) &
        (path_original['IVC'] >= 500.0) & (path_original['IVC'] <= 570.0) &
        (path_original['EVO'] >= 128.0) & (path_original['EVO'] <= 218.0) &
        (path_original['EVC'] >= 330.0) & (path_original['EVC'] <= 370.0)
    ) |
    (
        high_load &
        (path_original['IVO'] >= 345.0) & (path_original['IVO'] <= 365.0) &
        (path_original['IVC'] >= 495.0) & (path_original['IVC'] <= 535.0) &
        (path_original['EVO'] >= 128.0) & (path_original['EVO'] <= 218.0) &
        (path_original['EVC'] >= 345.0) & (path_original['EVC'] <= 355.0)
    )
)

path_original['Feasible'] = path_original['Load_ok'] & path_original['Boost_ok'] & path_original['BR_ok'] & path_original['VVA_ok']

# Compact display columns for decision making.
display_cols = [
    'Engine_speed', 'Boost pressure', 'Mass1', 'Mass2', 'SOI2', 'IVO', 'IVC', 'EVO', 'EVC',
    'Mass1+Mass2', 'Boost_low', 'Boost_high', 'Load_ok', 'Boost_ok', 'BR_ok', 'VVA_ok', 'Feasible'
]
path_original[display_cols]

In [ ]:
# 2D engineering constraint view: Boost vs (Mass1+Mass2) with explicit polygon boundaries
hist = pd_df.copy()
hist['Mass1+Mass2'] = hist['Mass1'] + hist['Mass2']

x = np.linspace(hist['Mass1+Mass2'].min(), max(40.0, hist['Mass1+Mass2'].max()), 400)
y_mid = 0.0922 * x + 0.8378
y_low = y_mid - 0.5
y_high = y_mid + 0.5

# Pull polygon caps from the current agent if available
min_load = float(getattr(agent, 'min_load', 3.0))
max_load = float(getattr(agent, 'load_limit', 30.0))
ambient_pressure = float(getattr(agent, 'ambient_pressure', 1.0))
tc_boost_limit = float(getattr(agent, 'tc_boost_limit', 3.8))

# Polygon is the intersection of: boost band, floor/roof, and left/right load walls
poly_low = np.maximum(y_low, ambient_pressure)
poly_high = np.minimum(y_high, tc_boost_limit)
poly_mask = (x >= min_load) & (x <= max_load) & (poly_high >= poly_low)

plt.figure(figsize=(10, 6))
plt.scatter(hist['Mass1+Mass2'], hist['Boost pressure'], s=25, c='#3b6fb6', alpha=0.65, label='Historical cases')

# Slanted boost corridor lines
plt.plot(x, y_mid, '--', c='gray', lw=1.5, label='Boost center line')
plt.plot(x, y_low, '--', c='red', lw=1.5, label='Boost lower slanted limit')
plt.plot(x, y_high, '--', c='red', lw=1.5, label='Boost upper slanted limit')

# Horizontal roof/floor
plt.axhline(ambient_pressure, color='purple', linestyle='-.', lw=1.4, label='Ambient pressure floor')
plt.axhline(tc_boost_limit, color='brown', linestyle='-.', lw=1.4, label='TC boost roof')

# Vertical load walls
plt.axvline(min_load, color='black', linestyle='-', lw=1.4, label='Min load wall')
plt.axvline(max_load, color='black', linestyle='-', lw=1.4, label='Max load wall')

# Filled feasible polygon region
plt.fill_between(
    x[poly_mask],
    poly_low[poly_mask],
    poly_high[poly_mask],
    color='limegreen',
    alpha=0.18,
    label='Feasible polygon region',
)

rec_x = path_original['Mass1+Mass2'].to_numpy()
rec_y = path_original['Boost pressure'].to_numpy()
plt.plot(rec_x, rec_y, '-o', c='orangered', lw=2.6, label='Recommended path')

for i, (xx, yy) in enumerate(zip(rec_x, rec_y), start=1):
    plt.text(xx + 0.25, yy + 0.02, f'Step {i}', color='orangered')

plt.xlabel('Mass1 + Mass2 [kg/h]')
plt.ylabel('Boost pressure [bar]')
plt.title('Constrained Planning Polygon and Recommended Steps')
plt.grid(alpha=0.25)
plt.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize candidate points from the exact planning run (no resampling)
if not hasattr(agent, 'last_plan_debug'):
    raise RuntimeError('No cached planner candidates found. Re-run Cell 7 first.')

dbg = agent.last_plan_debug
if 'candidate_sets_raw' not in dbg or len(dbg['candidate_sets_raw']) == 0:
    raise RuntimeError('Planner cache is empty. Re-run Cell 7 first.')

mass1_idx = input_features.index('Mass1')
boost_idx = input_features.index('Boost pressure')
step_idx = 0  # show candidate cloud for first planned step

sobol_step_scaled_blocks = []
local_step_scaled_blocks = []
for block in dbg['candidate_sets_raw']:
    sob = block['sobol_paths']
    loc = block['local_paths']

    if sob is not None and sob.numel() > 0:
        sobol_step_scaled_blocks.append(sob[:, step_idx, :].detach().cpu())
    if loc is not None and loc.numel() > 0:
        local_step_scaled_blocks.append(loc[:, step_idx, :].detach().cpu())

if not sobol_step_scaled_blocks:
    raise RuntimeError('No Sobol points found in planner cache.')

sobol_step_scaled = torch.cat(sobol_step_scaled_blocks, dim=0)
local_step_scaled = torch.cat(local_step_scaled_blocks, dim=0) if local_step_scaled_blocks else None

# Convert to physical units for plotting.
sobol_np = sobol_step_scaled.numpy()
sobol_m1 = scaler_x.inverse_transform(sobol_np[:, mass1_idx], mass1_idx)
sobol_bp = scaler_x.inverse_transform(sobol_np[:, boost_idx], boost_idx)

if local_step_scaled is not None:
    local_np = local_step_scaled.numpy()
    local_m1 = scaler_x.inverse_transform(local_np[:, mass1_idx], mass1_idx)
    local_bp = scaler_x.inverse_transform(local_np[:, boost_idx], boost_idx)
else:
    local_m1 = np.array([])
    local_bp = np.array([])

# Build a Mass1-vs-Boost projected feasible envelope using BR-band Mass2 limits.
min_load = float(getattr(agent, 'min_load', 3.0))
max_load = float(getattr(agent, 'load_limit', 30.0))
ambient_pressure = float(getattr(agent, 'ambient_pressure', 1.0))
tc_boost_limit = float(getattr(agent, 'tc_boost_limit', 3.8))
slope = float(getattr(agent, 'boost_slope', 0.0922))
intercept = float(getattr(agent, 'boost_intercept', 0.8378))
band = float(getattr(agent, 'boost_band', 0.5))

x = np.linspace(0.0, max_load, 500)

m2_low = np.piecewise(
    x,
    [(x < 10.0), (x >= 10.0) & (x < 20.0), (x >= 20.0) & (x <= 30.0)],
    [0.5, 0.9, 0.0],
)
m2_high = np.piecewise(
    x,
    [(x < 10.0), (x >= 10.0) & (x < 20.0), (x >= 20.0) & (x <= 30.0)],
    [3.5, 3.0, 1.5],
)

boost_low_env = np.maximum(slope * (x + m2_low) + intercept - band, ambient_pressure)
boost_high_env = np.minimum(slope * (x + m2_high) + intercept + band, tc_boost_limit)

# Keep only Sobol points that lie in the projected envelope.
low_at_sobol = np.interp(sobol_m1, x, boost_low_env, left=np.nan, right=np.nan)
high_at_sobol = np.interp(sobol_m1, x, boost_high_env, left=np.nan, right=np.nan)
sobol_projected_ok = (sobol_m1 >= 0.0) & (sobol_m1 <= max_load) & (sobol_bp >= low_at_sobol) & (sobol_bp <= high_at_sobol)
sobol_m1_plot = sobol_m1[sobol_projected_ok]
sobol_bp_plot = sobol_bp[sobol_projected_ok]

# Same projected mask for local points (optional, plotted faint).
if local_m1.size > 0:
    low_at_local = np.interp(local_m1, x, boost_low_env, left=np.nan, right=np.nan)
    high_at_local = np.interp(local_m1, x, boost_high_env, left=np.nan, right=np.nan)
    local_projected_ok = (local_m1 >= 0.0) & (local_m1 <= max_load) & (local_bp >= low_at_local) & (local_bp <= high_at_local)
    local_m1_plot = local_m1[local_projected_ok]
    local_bp_plot = local_bp[local_projected_ok]
else:
    local_m1_plot = np.array([])
    local_bp_plot = np.array([])

plt.figure(figsize=(10, 6))
plt.fill_between(x, boost_low_env, boost_high_env, color='limegreen', alpha=0.14, label='Projected feasible envelope')
plt.plot(x, boost_low_env, '--', c='crimson', lw=1.3, label='Projected lower/upper limits')
plt.plot(x, boost_high_env, '--', c='crimson', lw=1.3)
plt.axhline(ambient_pressure, color='purple', linestyle='-.', lw=1.2, label='Ambient floor')
plt.axhline(tc_boost_limit, color='brown', linestyle='-.', lw=1.2, label='TC roof')
plt.axvline(max_load, color='black', linestyle='-', lw=1.2, label='Load wall')

plt.scatter(sobol_m1_plot, sobol_bp_plot, s=8, c='gray', alpha=0.25, label='Sobol candidates (projected-feasible)')
if local_m1_plot.size > 0:
    plt.scatter(local_m1_plot, local_bp_plot, s=8, c='orange', alpha=0.60, label='Local RW candidates (projected-feasible)')

plt.scatter(path_original['Mass1'], path_original['Boost pressure'], s=65, c='orangered', edgecolors='black', linewidths=0.7, label='Recommended path')

plt.xlabel('Mass1 [kg/h]')
plt.ylabel('Boost pressure [bar]')
plt.title('Actual Candidate Cloud Used for Next-Experiment Planning')
plt.grid(alpha=0.25)
plt.legend(loc='best')
plt.tight_layout()
plt.show()

print(f"Sobol points shown (projected-feasible): {sobol_m1_plot.shape[0]} / {sobol_m1.shape[0]}")
print(f"Local RW points shown (projected-feasible): {local_m1_plot.shape[0]} / {local_m1.shape[0]}")
print(f"feasible_sobol_enabled: {dbg.get('enforce_feasible_sobol', False)}")
print(f"include_local_paths: {dbg.get('include_local_paths', 'n/a')}")
print(f"local_path_fraction: {dbg.get('local_path_fraction', 'n/a')}")
print(f"feasible_margin_weight: {dbg.get('feasible_margin_weight', 'n/a')}")

In [ ]:
# 3D mean/variance recommendations plot with constrained-region overlay
import matplotlib.gridspec as gridspec


def plot_recommendations_with_constraints(
    agent,
    optimal_path,
    bounds,
    feature_names,
    scaler_x,
    output_index=0,
    res=90,
    fixed_point_mode="current",      # "current" | "zero" | "custom"
    custom_fixed_original=None,       # dict like {"Mass2": 1.2, "SOI2": 45.0}
    surface_mode="mean",             # "mean" | "sample"
    sample_seed=42,
):
    """Plot 1x2 GP surfaces with engineering constraints and style toggles."""
    required = {"Mass1", "Mass2", "Boost pressure"}
    missing = required.difference(set(feature_names))
    if missing:
        raise ValueError(f"Required constrained features missing: {missing}")

    mass1_idx = feature_names.index("Mass1")
    boost_idx = feature_names.index("Boost pressure")

    if output_index < 0 or output_index >= len(agent.models):
        raise ValueError(f"output_index must be in [0, {len(agent.models)-1}]")

    # Handle bounds layout robustly: [d,2] or [2,d].
    if bounds.shape[1] == 2:
        lower = bounds[:, 0]
        upper = bounds[:, 1]
    elif bounds.shape[0] == 2:
        lower = bounds[0, :]
        upper = bounds[1, :]
    else:
        raise ValueError(f"Unexpected bounds shape: {tuple(bounds.shape)}")

    # Convert plotted bounds (scaled -> physical) for the two axes.
    m1_min = float(scaler_x.inverse_transform(lower[mass1_idx].item(), mass1_idx))
    m1_max = float(scaler_x.inverse_transform(upper[mass1_idx].item(), mass1_idx))
    bp_min = float(scaler_x.inverse_transform(lower[boost_idx].item(), boost_idx))
    bp_max = float(scaler_x.inverse_transform(upper[boost_idx].item(), boost_idx))

    M1, BP = np.meshgrid(
        np.linspace(m1_min, m1_max, res),
        np.linspace(bp_min, bp_max, res),
        indexing="xy",
    )

    # Base point for non-plotted dimensions.
    if fixed_point_mode == "current":
        base_point = agent.X[-1].detach().clone()
    elif fixed_point_mode == "zero":
        base_point = torch.zeros(agent.d, dtype=torch.float64, device=agent.X.device)
    elif fixed_point_mode == "custom":
        base_point = agent.X[-1].detach().clone()
        if custom_fixed_original is None:
            raise ValueError("custom_fixed_original must be provided when fixed_point_mode='custom'")
        for k, v in custom_fixed_original.items():
            if k not in feature_names:
                raise ValueError(f"Unknown feature in custom_fixed_original: {k}")
            idx = feature_names.index(k)
            mu = float(scaler_x.scaler.mean_[idx])
            sig = float(scaler_x.scaler.scale_[idx])
            base_point[idx] = (float(v) - mu) / sig
    else:
        raise ValueError("fixed_point_mode must be one of: 'current', 'zero', 'custom'")

    query = base_point.repeat(res * res, 1)

    m1_scaled = (
        torch.tensor(M1.reshape(-1), dtype=torch.float64)
        - torch.tensor(scaler_x.scaler.mean_[mass1_idx], dtype=torch.float64)
    ) / torch.tensor(scaler_x.scaler.scale_[mass1_idx], dtype=torch.float64)
    bp_scaled = (
        torch.tensor(BP.reshape(-1), dtype=torch.float64)
        - torch.tensor(scaler_x.scaler.mean_[boost_idx], dtype=torch.float64)
    ) / torch.tensor(scaler_x.scaler.scale_[boost_idx], dtype=torch.float64)

    query[:, mass1_idx] = m1_scaled
    query[:, boost_idx] = bp_scaled

    with torch.no_grad():
        post = agent.models[output_index].posterior(query)
        if surface_mode == "mean":
            mean = post.mean.squeeze(-1).cpu().numpy().reshape(res, res)
        elif surface_mode == "sample":
            torch.manual_seed(int(sample_seed))
            mean = post.rsample(torch.Size([1])).squeeze(0).squeeze(-1).cpu().numpy().reshape(res, res)
        else:
            raise ValueError("surface_mode must be 'mean' or 'sample'")

        var_parts = [
            mdl.posterior(query).variance.squeeze(-1).cpu().numpy()
            for mdl in agent.models
        ]
        var = np.sum(np.stack(var_parts, axis=0), axis=0).reshape(res, res)

    # Historical points in physical units.
    hist_m1 = np.array([scaler_x.inverse_transform(v, mass1_idx) for v in agent.X[:, mass1_idx].detach().cpu().numpy()])
    hist_bp = np.array([scaler_x.inverse_transform(v, boost_idx) for v in agent.X[:, boost_idx].detach().cpu().numpy()])
    hist_y = agent.Y[:, output_index].detach().cpu().numpy().reshape(-1)

    # Planned path in physical units.
    path_scaled_np = optimal_path.detach().cpu().numpy()
    path_m1 = np.array([scaler_x.inverse_transform(v, mass1_idx) for v in path_scaled_np[:, mass1_idx]])
    path_bp = np.array([scaler_x.inverse_transform(v, boost_idx) for v in path_scaled_np[:, boost_idx]])
    curr_m1 = float(scaler_x.inverse_transform(agent.X[-1, mass1_idx].item(), mass1_idx))
    curr_bp = float(scaler_x.inverse_transform(agent.X[-1, boost_idx].item(), boost_idx))

    # Constraint envelope in Mass1-vs-Boost projection.
    min_load = float(getattr(agent, "min_load", 3.0))
    max_load = float(getattr(agent, "load_limit", 30.0))
    ambient_pressure = float(getattr(agent, "ambient_pressure", 1.0))
    tc_boost_limit = float(getattr(agent, "tc_boost_limit", 3.8))
    slope = float(getattr(agent, "boost_slope", 0.0922))
    intercept = float(getattr(agent, "boost_intercept", 0.8378))
    band = float(getattr(agent, "boost_band", 0.5))

    cx = np.linspace(max(0.0, m1_min), min(max_load, m1_max), 400)
    m2_low = np.piecewise(
        cx,
        [(cx < 10.0), (cx >= 10.0) & (cx < 20.0), (cx >= 20.0) & (cx <= 30.0)],
        [0.5, 0.9, 0.0],
    )
    m2_high = np.piecewise(
        cx,
        [(cx < 10.0), (cx >= 10.0) & (cx < 20.0), (cx >= 20.0) & (cx <= 30.0)],
        [3.5, 3.0, 1.5],
    )
    boost_low = np.maximum(slope * (cx + m2_low) + intercept - band, ambient_pressure)
    boost_high = np.minimum(slope * (cx + m2_high) + intercept + band, tc_boost_limit)
    c_mask = (cx >= min_load) & (cx <= max_load) & (boost_high >= boost_low)
    cx, boost_low, boost_high = cx[c_mask], boost_low[c_mask], boost_high[c_mask]

    fig = plt.figure(figsize=(13, 6), constrained_layout=True)
    gs = gridspec.GridSpec(1, 2, figure=fig)
    ax_mean = fig.add_subplot(gs[0, 0], projection="3d")
    ax_var = fig.add_subplot(gs[0, 1], projection="3d")

    mean_surf = ax_mean.plot_surface(M1, BP, mean, cmap="viridis", alpha=0.55, edgecolor="none")
    ax_mean.scatter(hist_m1, hist_bp, hist_y, c="k", s=18, alpha=0.45, label="Historical")

    var_surf = ax_var.plot_surface(M1, BP, var, cmap="plasma", alpha=0.62, edgecolor="none")

    full_path_x = np.concatenate([[curr_m1], path_m1])
    full_path_y = np.concatenate([[curr_bp], path_bp])
    path_z = np.full_like(full_path_x, float(np.nanmax(var) * 0.98))
    ax_var.plot(full_path_x, full_path_y, path_z, c="darkgreen", linestyle="--", linewidth=2.0, label="Suggested path")
    ax_var.scatter(path_m1, path_bp, np.full_like(path_m1, float(np.nanmax(var) * 0.98)), c="orange", s=90, edgecolors="k", label="Suggested experiments")
    ax_var.scatter(path_m1[0], path_bp[0], float(np.nanmax(var) * 0.98), c="red", marker="*", s=250, edgecolors="none", label="Do this next")

    def _draw_constraints(ax, zmin, zmax):
        z_floor = zmin + 0.02 * (zmax - zmin)
        Xc = np.vstack([cx, cx])
        Yc = np.vstack([boost_low, boost_high])
        Zc = np.full_like(Xc, z_floor)
        ax.plot_surface(Xc, Yc, Zc, color="limegreen", alpha=0.22, edgecolor="none")
        ax.plot(cx, boost_low, np.full_like(cx, zmax), color="crimson", linestyle="--", linewidth=1.8, label="Constraint envelope")
        ax.plot(cx, boost_high, np.full_like(cx, zmax), color="crimson", linestyle="--", linewidth=1.8)
        for wall in [min_load, max_load]:
            if wall >= m1_min and wall <= m1_max:
                ax.plot([wall, wall], [bp_min, bp_min], [zmin, zmax], color="black", linewidth=1.1, alpha=0.85)
                ax.plot([wall, wall], [bp_max, bp_max], [zmin, zmax], color="black", linewidth=1.1, alpha=0.85)

    _draw_constraints(ax_mean, float(np.nanmin(mean)), float(np.nanmax(mean)))
    _draw_constraints(ax_var, float(np.nanmin(var)), float(np.nanmax(var)))

    title_mode = f"{surface_mode}, fixed={fixed_point_mode}, output={output_index}, res={res}"
    ax_mean.set_title(f"Predictive Surface\nMass1 vs Boost pressure ({title_mode})")
    ax_var.set_title("Predictive Variance Surface\nwith constrained suggested steps")

    for ax in [ax_mean, ax_var]:
        ax.view_init(elev=34, azim=-48)
        ax.set_xlabel("Mass1 [kg/h]")
        ax.set_ylabel("Boost pressure [bar]")
        ax.set_facecolor("white")
        ax.xaxis.pane.fill = False
        ax.yaxis.pane.fill = False
        ax.zaxis.pane.fill = False
        ax.legend(loc="upper left")

    fig.colorbar(mean_surf, ax=ax_mean, shrink=0.68, pad=0.06, label="Predictive value")
    fig.colorbar(var_surf, ax=ax_var, shrink=0.68, pad=0.06, label="Total predictive variance")
    plt.show()


def plot_output_sweep_with_constraints(
    agent,
    optimal_path,
    bounds,
    feature_names,
    scaler_x,
    output_indices=(0, 1, 2),
    res=70,
    fixed_point_mode="current",
    surface_mode="mean",
):
    """Quick compare of several outputs to find where waviness appears."""
    n = len(output_indices)
    fig = plt.figure(figsize=(5.5 * n, 4.6), constrained_layout=True)

    # Minimal local helper for shared 2D slice generation.
    if bounds.shape[1] == 2:
        lower = bounds[:, 0]
        upper = bounds[:, 1]
    else:
        lower = bounds[0, :]
        upper = bounds[1, :]

    mass1_idx = feature_names.index("Mass1")
    boost_idx = feature_names.index("Boost pressure")
    m1_min = float(scaler_x.inverse_transform(lower[mass1_idx].item(), mass1_idx))
    m1_max = float(scaler_x.inverse_transform(upper[mass1_idx].item(), mass1_idx))
    bp_min = float(scaler_x.inverse_transform(lower[boost_idx].item(), boost_idx))
    bp_max = float(scaler_x.inverse_transform(upper[boost_idx].item(), boost_idx))
    M1, BP = np.meshgrid(np.linspace(m1_min, m1_max, res), np.linspace(bp_min, bp_max, res), indexing="xy")

    if fixed_point_mode == "current":
        base_point = agent.X[-1].detach().clone()
    else:
        base_point = torch.zeros(agent.d, dtype=torch.float64, device=agent.X.device)

    query = base_point.repeat(res * res, 1)
    m1_scaled = (
        torch.tensor(M1.reshape(-1), dtype=torch.float64)
        - torch.tensor(scaler_x.scaler.mean_[mass1_idx], dtype=torch.float64)
    ) / torch.tensor(scaler_x.scaler.scale_[mass1_idx], dtype=torch.float64)
    bp_scaled = (
        torch.tensor(BP.reshape(-1), dtype=torch.float64)
        - torch.tensor(scaler_x.scaler.mean_[boost_idx], dtype=torch.float64)
    ) / torch.tensor(scaler_x.scaler.scale_[boost_idx], dtype=torch.float64)
    query[:, mass1_idx] = m1_scaled
    query[:, boost_idx] = bp_scaled

    for k, out_idx in enumerate(output_indices, start=1):
        ax = fig.add_subplot(1, n, k, projection="3d")
        with torch.no_grad():
            post = agent.models[int(out_idx)].posterior(query)
            if surface_mode == "sample":
                torch.manual_seed(42 + int(out_idx))
                Z = post.rsample(torch.Size([1])).squeeze(0).squeeze(-1).cpu().numpy().reshape(res, res)
            else:
                Z = post.mean.squeeze(-1).cpu().numpy().reshape(res, res)
        surf = ax.plot_surface(M1, BP, Z, cmap="viridis", alpha=0.82, edgecolor="none")
        ax.set_title(f"Output {int(out_idx)} ({surface_mode})")
        ax.set_xlabel("Mass1")
        ax.set_ylabel("Boost")
        ax.set_zlabel("Pred")
        ax.view_init(elev=32, azim=-48)
        fig.colorbar(surf, ax=ax, shrink=0.72, pad=0.04)

    plt.show()


# Main constrained plot: default smooth view
plot_recommendations_with_constraints(
    agent=agent,
    optimal_path=path_scaled,
    bounds=scaled_bounds,
    feature_names=input_features,
    scaler_x=scaler_x,
    output_index=0,
    res=90,
    fixed_point_mode="current",
    surface_mode="mean",
)

# Wavy-style view (uncomment to compare):
# plot_recommendations_with_constraints(
#     agent=agent,
#     optimal_path=path_scaled,
#     bounds=scaled_bounds,
#     feature_names=input_features,
#     scaler_x=scaler_x,
#     output_index=0,
#     res=120,
#     fixed_point_mode="zero",
#     surface_mode="sample",
#     sample_seed=7,
# )

# Output sweep (uncomment to compare outputs quickly):
# plot_output_sweep_with_constraints(
#     agent=agent,
#     optimal_path=path_scaled,
#     bounds=scaled_bounds,
#     feature_names=input_features,
#     scaler_x=scaler_x,
#     output_indices=(0, 1, 2),
#     res=80,
#     fixed_point_mode="current",
#     surface_mode="mean",
# )


In [ ]:
# Wavy-style comparison view
plot_recommendations_with_constraints(
    agent=agent,
    optimal_path=path_scaled,
    bounds=scaled_bounds,
    feature_names=input_features,
    scaler_x=scaler_x,
    output_index=0,
    res=120,
    fixed_point_mode="zero",
    surface_mode="sample",
    sample_seed=7,
)

In [ ]:
# Waviness diagnostic: auto-pick roughest output and show mean/sample/amplified sample

def _surface_slice_for_output(agent, scaler_x, feature_names, bounds, out_idx, res=120, fixed_mode="current", seed=42):
    mass1_idx = feature_names.index("Mass1")
    boost_idx = feature_names.index("Boost pressure")

    if bounds.shape[1] == 2:
        lower = bounds[:, 0]
        upper = bounds[:, 1]
    else:
        lower = bounds[0, :]
        upper = bounds[1, :]

    m1_min = float(scaler_x.inverse_transform(lower[mass1_idx].item(), mass1_idx))
    m1_max = float(scaler_x.inverse_transform(upper[mass1_idx].item(), mass1_idx))
    bp_min = float(scaler_x.inverse_transform(lower[boost_idx].item(), boost_idx))
    bp_max = float(scaler_x.inverse_transform(upper[boost_idx].item(), boost_idx))

    M1, BP = np.meshgrid(
        np.linspace(m1_min, m1_max, res),
        np.linspace(bp_min, bp_max, res),
        indexing="xy",
    )

    if fixed_mode == "current":
        base_point = agent.X[-1].detach().clone()
    else:
        base_point = torch.zeros(agent.d, dtype=torch.float64, device=agent.X.device)

    query = base_point.repeat(res * res, 1)
    m1_scaled = (
        torch.tensor(M1.reshape(-1), dtype=torch.float64)
        - torch.tensor(scaler_x.scaler.mean_[mass1_idx], dtype=torch.float64)
    ) / torch.tensor(scaler_x.scaler.scale_[mass1_idx], dtype=torch.float64)
    bp_scaled = (
        torch.tensor(BP.reshape(-1), dtype=torch.float64)
        - torch.tensor(scaler_x.scaler.mean_[boost_idx], dtype=torch.float64)
    ) / torch.tensor(scaler_x.scaler.scale_[boost_idx], dtype=torch.float64)
    query[:, mass1_idx] = m1_scaled
    query[:, boost_idx] = bp_scaled

    with torch.no_grad():
        post = agent.models[int(out_idx)].posterior(query)
        mean = post.mean.squeeze(-1).cpu().numpy().reshape(res, res)
        torch.manual_seed(int(seed) + int(out_idx))
        sample = post.rsample(torch.Size([1])).squeeze(0).squeeze(-1).cpu().numpy().reshape(res, res)

    return M1, BP, mean, sample


def _roughness_score(z):
    d2x = np.diff(z, n=2, axis=1)
    d2y = np.diff(z, n=2, axis=0)
    return float(np.std(d2x) + np.std(d2y))


# Find which output has the strongest curvature on a current-point slice.
roughness = []
for oi in range(len(agent.models)):
    _, _, mean_oi, _ = _surface_slice_for_output(
        agent, scaler_x, input_features, scaled_bounds, out_idx=oi, res=90, fixed_mode="current", seed=7
    )
    roughness.append(_roughness_score(mean_oi))

best_out = int(np.argmax(roughness))
print("Roughness by output index:", [round(v, 6) for v in roughness])
print("Most wavy output index on current-point slice:", best_out)

# Build visualization for the selected output.
M1, BP, mean_z, sample_z = _surface_slice_for_output(
    agent, scaler_x, input_features, scaled_bounds, out_idx=best_out, res=120, fixed_mode="current", seed=7
)

# Amplified sample keeps GP sample shape but scales deviations from mean for visibility.
amplify = 6.0
sample_amp = mean_z + amplify * (sample_z - mean_z)

fig = plt.figure(figsize=(18, 5.5), constrained_layout=True)
ax1 = fig.add_subplot(1, 3, 1, projection="3d")
ax2 = fig.add_subplot(1, 3, 2, projection="3d")
ax3 = fig.add_subplot(1, 3, 3, projection="3d")

s1 = ax1.plot_surface(M1, BP, mean_z, cmap="viridis", alpha=0.9, edgecolor="none")
ax1.set_title(f"Mean surface (output {best_out})")
ax1.set_xlabel("Mass1 [kg/h]")
ax1.set_ylabel("Boost pressure [bar]")
ax1.set_zlabel("Prediction")
ax1.view_init(elev=32, azim=-48)
fig.colorbar(s1, ax=ax1, shrink=0.72, pad=0.04)

s2 = ax2.plot_surface(M1, BP, sample_z, cmap="viridis", alpha=0.9, edgecolor="none")
ax2.set_title(f"Posterior sample (output {best_out})")
ax2.set_xlabel("Mass1 [kg/h]")
ax2.set_ylabel("Boost pressure [bar]")
ax2.set_zlabel("Prediction")
ax2.view_init(elev=32, azim=-48)
fig.colorbar(s2, ax=ax2, shrink=0.72, pad=0.04)

s3 = ax3.plot_surface(M1, BP, sample_amp, cmap="viridis", alpha=0.9, edgecolor="none")
ax3.set_title(f"Amplified sample x{amplify:.1f} (visual diagnostic)")
ax3.set_xlabel("Mass1 [kg/h]")
ax3.set_ylabel("Boost pressure [bar]")
ax3.set_zlabel("Prediction")
ax3.view_init(elev=32, azim=-48)
fig.colorbar(s3, ax=ax3, shrink=0.72, pad=0.04)

plt.show()


In [ ]:
# Final visualization: boundary representation on the surface + variance reduction plot
# Assumes agent, path_scaled, path_original, hist, current_loc are already available.

# ------------------------------
# 1) Boundary representation on the boost surface
# ------------------------------
hist_vis = pd_df.copy()
hist_vis['Mass1+Mass2'] = hist_vis['Mass1'] + hist_vis['Mass2']

x_grid = np.linspace(hist_vis['Mass1+Mass2'].min(), max(40.0, hist_vis['Mass1+Mass2'].max()), 220)
y_center = 0.0922 * x_grid + 0.8378
y_low = y_center - 0.5
y_high = y_center + 0.5

plt.figure(figsize=(11, 7))
plt.scatter(
    hist_vis['Mass1+Mass2'],
    hist_vis['Boost pressure'],
    s=22,
    c='#4169e1',
    alpha=0.55,
    label='Historical cases',
)
plt.plot(x_grid, y_center, color='gray', linestyle='--', linewidth=1.6, label='Boost center')
plt.plot(x_grid, y_low, color='crimson', linestyle='--', linewidth=1.6, label='Boost lower/upper band')
plt.plot(x_grid, y_high, color='crimson', linestyle='--', linewidth=1.6)
plt.axvline(30.0, color='crimson', linestyle='-', linewidth=1.4, label='Load limit')

# Highlight recommended path.
rec_mass = path_original['Mass1+Mass2'].to_numpy()
rec_boost = path_original['Boost pressure'].to_numpy()
plt.plot(rec_mass, rec_boost, '-o', color='orangered', linewidth=2.6, markersize=6, label='Recommended path')

# Shade the feasible corridor up to the load limit.
feasible_mask = x_grid <= 30.0
plt.fill_between(
    x_grid[feasible_mask],
    y_low[feasible_mask],
    y_high[feasible_mask],
    color='green',
    alpha=0.10,
    label='Feasible boost corridor',
)

plt.xlabel('Mass1 + Mass2 [kg/h]')
plt.ylabel('Boost pressure [bar]')
plt.title('Surface Boundary Representation in the Boost Domain')
plt.grid(alpha=0.25)
plt.legend(loc='best')
plt.tight_layout()
plt.show()

# ------------------------------
# 2) Variance reduction plot
# ------------------------------
# Compare mean predictive variance at the current location versus along the planned path.
current_repeat = current_loc.repeat(path_scaled.shape[0], 1)

current_var = []
planned_var = []
for model, likelihood in zip(agent.models, agent.likelihoods):
    model.eval()
    likelihood.eval()
    with torch.no_grad():
        current_pred = model.posterior(current_repeat)
        planned_pred = model.posterior(path_scaled)
        current_var.append(current_pred.variance.squeeze(-1).cpu().numpy())
        planned_var.append(planned_pred.variance.squeeze(-1).cpu().numpy())

current_var = np.stack(current_var, axis=0).mean(axis=0)
planned_var = np.stack(planned_var, axis=0).mean(axis=0)
variance_reduction = current_var - planned_var

step_labels = [f'Step {i + 1}' for i in range(len(variance_reduction))]

plt.figure(figsize=(10, 5))
plt.plot(step_labels, current_var, marker='o', linewidth=2.0, label='Current-state variance baseline')
plt.plot(step_labels, planned_var, marker='o', linewidth=2.0, label='Planned-path variance')
plt.bar(step_labels, variance_reduction, alpha=0.25, color='seagreen', label='Variance reduction')
plt.axhline(0.0, color='black', linewidth=1.0)
plt.ylabel('Mean predictive variance')
plt.title('Variance Reduction Along the Recommended Path')
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
# Boundary views for the constrained operating region
# This cell visualizes the same latest polygon constraints used by the planner/2D boundary plot.

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Use the original physical-scale data for boundary visualization.
vis_df = pd_df.copy()
vis_df['Mass1+Mass2'] = vis_df['Mass1'] + vis_df['Mass2']
prop_df = path_original.copy()
prop_df['Mass1+Mass2'] = prop_df['Mass1'] + prop_df['Mass2']

# Pull current live boundary settings from the fitted/loaded agent.
min_load = float(getattr(agent, 'min_load', 3.0))
max_load = float(getattr(agent, 'load_limit', 30.0))
ambient_pressure = float(getattr(agent, 'ambient_pressure', 1.0))
tc_boost_limit = float(getattr(agent, 'tc_boost_limit', 3.8))
boost_slope = float(getattr(agent, 'boost_slope', 0.0922))
boost_intercept = float(getattr(agent, 'boost_intercept', 0.8378))
boost_band = float(getattr(agent, 'boost_band', 0.5))

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# -------------------------------------------------
# 1) Boost polygon boundary in Mass1+Mass2 vs Boost pressure
# -------------------------------------------------
ax = axes[0]
x = np.linspace(vis_df['Mass1+Mass2'].min(), max(40.0, vis_df['Mass1+Mass2'].max()), 300)
y_center = boost_slope * x + boost_intercept
y_low = y_center - boost_band
y_high = y_center + boost_band

poly_low = np.maximum(y_low, ambient_pressure)
poly_high = np.minimum(y_high, tc_boost_limit)
poly_mask = (x >= min_load) & (x <= max_load) & (poly_high >= poly_low)

ax.scatter(vis_df['Mass1+Mass2'], vis_df['Boost pressure'], s=14, c='#3b6fb6', alpha=0.45, label='Data')
ax.scatter(
    prop_df['Mass1+Mass2'],
    prop_df['Boost pressure'],
    s=60,
    c='orangered',
    edgecolors='black',
    linewidths=0.6,
    label='Proposed experiments',
)

# Slanted corridor
ax.plot(x, y_center, '--', c='gray', lw=1.5, label='Boost center line')
ax.plot(x, y_low, '--', c='crimson', lw=1.5, label='Slanted lower/upper')
ax.plot(x, y_high, '--', c='crimson', lw=1.5)

# Floor/roof and load walls
ax.axhline(ambient_pressure, c='purple', lw=1.2, linestyle='-.', label='Ambient floor')
ax.axhline(tc_boost_limit, c='brown', lw=1.2, linestyle='-.', label='TC roof')
ax.axvline(min_load, c='black', lw=1.2, linestyle='-', label='Min load wall')
ax.axvline(max_load, c='black', lw=1.2, linestyle='-', label='Max load wall')

# Polygon fill
ax.fill_between(
    x[poly_mask],
    poly_low[poly_mask],
    poly_high[poly_mask],
    color='green',
    alpha=0.10,
    label='Feasible polygon',
)

ax.set_xlabel('Mass1 + Mass2 [kg/h]')
ax.set_ylabel('Boost pressure [bar]')
ax.set_title('Boost Polygon Boundary (Latest)')
ax.grid(alpha=0.25)
ax.legend(loc='best', fontsize=8)

# -------------------------------------------------
# 2) BR boundary in Mass1 vs Mass2
# -------------------------------------------------
ax = axes[1]
ax.scatter(vis_df['Mass1'], vis_df['Mass2'], s=14, c='#4169e1', alpha=0.45, label='Data')
ax.scatter(
    prop_df['Mass1'],
    prop_df['Mass2'],
    s=60,
    c='orangered',
    edgecolors='black',
    linewidths=0.6,
    label='Proposed experiments',
)

# BR load-band limits kept consistent with planner settings.
br_boxes = [
    (0.0, 10.0, 0.5, 3.5, 'tab:green', '0 < Mass1 < 10'),
    (10.0, 20.0, 0.9, 3.0, 'tab:orange', '10 <= Mass1 < 20'),
    (20.0, 30.0, 0.0, 1.5, 'tab:red', '20 <= Mass1 < 30'),
]

for x0, x1, y0, y1, color, label in br_boxes:
    ax.plot([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0], color=color, lw=2.0, label=label)
    ax.fill_between([x0, x1], [y0, y0], [y1, y1], color=color, alpha=0.08)

ax.set_xlim(0, 35)
ax.set_ylim(0, 5)
ax.set_xlabel('Mass1 [kg/h]')
ax.set_ylabel('Mass2 [kg/h]')
ax.set_title('BR Boundary')
ax.grid(alpha=0.25)
ax.legend(loc='best', fontsize=8)

# -------------------------------------------------
# 3) VVA boundary in Mass1 vs valve timing bands
# -------------------------------------------------
ax = axes[2]
ax.scatter(vis_df['Mass1'], vis_df['IVO'], s=10, c='#6a5acd', alpha=0.30, label='IVO data')
ax.scatter(vis_df['Mass1'], vis_df['IVC'], s=10, c='#ff8c00', alpha=0.30, label='IVC data')
ax.scatter(vis_df['Mass1'], vis_df['EVO'], s=10, c='#2e8b57', alpha=0.30, label='EVO data')
ax.scatter(vis_df['Mass1'], vis_df['EVC'], s=10, c='#daa520', alpha=0.30, label='EVC data')

# Keep one consistent proposed-point style across all VVA parameters.
proposed_vva_style = dict(s=60, c='orangered', edgecolors='black', linewidths=0.6)
ax.scatter(prop_df['Mass1'], prop_df['IVO'], label='Proposed VVA points', **proposed_vva_style)
ax.scatter(prop_df['Mass1'], prop_df['IVC'], label='_nolegend_', **proposed_vva_style)
ax.scatter(prop_df['Mass1'], prop_df['EVO'], label='_nolegend_', **proposed_vva_style)
ax.scatter(prop_df['Mass1'], prop_df['EVC'], label='_nolegend_', **proposed_vva_style)

# VVA ranges by load band, aligned with planner constraints.
load_bands = [
    (0.0, 10.0, {'IVO': (350.0, 435.0), 'IVC': (500.0, 540.0), 'EVO': (128.0, 218.0), 'EVC': (270.0, 350.0)}, '0 < Mass1 < 10'),
    (10.0, 20.0, {'IVO': (330.0, 390.0), 'IVC': (500.0, 570.0), 'EVO': (128.0, 218.0), 'EVC': (330.0, 370.0)}, '10 <= Mass1 < 20'),
    (20.0, 30.0, {'IVO': (345.0, 365.0), 'IVC': (495.0, 535.0), 'EVO': (128.0, 218.0), 'EVC': (345.0, 355.0)}, '20 <= Mass1 < 30'),
]

# Color each boundary pair by VVA parameter for easier reading.
vva_param_colors = {
    'IVO': '#7b61ff',
    'IVC': '#ff8c00',
    'EVO': '#2e8b57',
    'EVC': '#c9a227',
}

for x0, x1, limits, label in load_bands:
    for name, (lo, hi) in limits.items():
        c = vva_param_colors.get(name, 'gray')
        ax.plot([x0, x1], [lo, lo], linestyle='--', lw=1.2, alpha=0.85, color=c)
        ax.plot([x0, x1], [hi, hi], linestyle='--', lw=1.2, alpha=0.85, color=c)
    ax.axvspan(x0, x1, alpha=0.04, color='gray')
    ax.text((x0 + x1) / 2, 710, label, ha='center', va='top', fontsize=8)

# Split legend into two compact blocks: markers and limits.
vva_marker_handles = [
    Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#6a5acd', markeredgecolor='none', markersize=5, alpha=0.30, label='IVO data'),
    Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#ff8c00', markeredgecolor='none', markersize=5, alpha=0.30, label='IVC data'),
    Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#2e8b57', markeredgecolor='none', markersize=5, alpha=0.30, label='EVO data'),
    Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='#daa520', markeredgecolor='none', markersize=5, alpha=0.30, label='EVC data'),
    Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='orangered', markeredgecolor='black', markersize=7, label='Proposed VVA points'),
]
vva_boundary_handles = [
    Line2D([0], [0], color=vva_param_colors['IVO'], lw=1.5, linestyle='--', label='IVO limits'),
    Line2D([0], [0], color=vva_param_colors['IVC'], lw=1.5, linestyle='--', label='IVC limits'),
    Line2D([0], [0], color=vva_param_colors['EVO'], lw=1.5, linestyle='--', label='EVO limits'),
    Line2D([0], [0], color=vva_param_colors['EVC'], lw=1.5, linestyle='--', label='EVC limits'),
]

legend_markers = ax.legend(
    handles=vva_marker_handles,
    loc='upper left',
    bbox_to_anchor=(1.01, 1.0),
    borderaxespad=0.0,
    fontsize=7,
    ncol=1,
    title='VVA points',
    title_fontsize=7,
 )
ax.add_artist(legend_markers)
ax.legend(
    handles=vva_boundary_handles,
    loc='upper left',
    bbox_to_anchor=(1.01, 0.50),
    borderaxespad=0.0,
    fontsize=7,
    ncol=1,
    title='VVA limits',
    title_fontsize=7,
 )

ax.set_xlim(0, 30)
ax.set_ylim(0, 720)
ax.set_xlabel('Mass1 [kg/h]')
ax.set_ylabel('Valve timing [CAD]')
ax.set_title('VVA Boundary')
ax.grid(alpha=0.25)

plt.tight_layout(rect=[0, 0, 0.87, 1])
plt.show()

In [ ]:
# Information gain along the recommended path
# This plot uses the same posterior covariance calculation as the constrained agent.

def _path_information_gain(agent, path_scaled):
    q_steps = path_scaled.shape[0]
    cumulative_ig = np.zeros(q_steps, dtype=float)

    for model, likelihood in zip(agent.models, agent.likelihoods):
        model.eval()
        likelihood.eval()
        noise = float(likelihood.noise.clamp_min(1e-8).item())

        with torch.no_grad():
            for step in range(1, q_steps + 1):
                prefix = path_scaled[:step].unsqueeze(0)
                posterior = model.posterior(prefix)
                cov = posterior.distribution.covariance_matrix
                if cov.ndim == 3:
                    cov = cov.squeeze(0)

                eye = torch.eye(step, dtype=cov.dtype, device=cov.device)
                m = eye + cov / noise

                try:
                    chol = torch.linalg.cholesky(m)
                    ig_step = chol.diagonal(dim1=-2, dim2=-1).log().sum(dim=-1)
                except RuntimeError:
                    ig_step = 0.5 * torch.linalg.slogdet(m)[1]

                cumulative_ig[step - 1] += float(ig_step.item())

    incremental_ig = np.diff(np.concatenate(([0.0], cumulative_ig)))
    return cumulative_ig, incremental_ig

cumulative_ig, incremental_ig = _path_information_gain(agent, path_scaled)
step_labels = [f'Step {i + 1}' for i in range(len(cumulative_ig))]

plt.figure(figsize=(10, 5))
plt.bar(step_labels, incremental_ig, alpha=0.35, color='steelblue', label='Incremental IG')
plt.plot(step_labels, cumulative_ig, marker='o', linewidth=2.0, color='navy', label='Cumulative IG')
plt.axhline(0.0, color='black', linewidth=1.0)
plt.ylabel('Information gain')
plt.title('Information Gain Along the Recommended Path')
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
# Mass1 vs Boost pressure: three 3D variants
# 1) Historical surface only
# 2) Historical surface + boundaries + proposed points
# 3) Boundary surface + historical markers + proposed points

# Physical-scale data
hist_mass1 = pd_df['Mass1'].to_numpy()
hist_boost = pd_df['Boost pressure'].to_numpy()
prop_mass1 = path_original['Mass1'].to_numpy()
prop_boost = path_original['Boost pressure'].to_numpy()

# Common grid in Mass1/Boost plane
mass1_vals = np.linspace(0.0, 30.0, 220)
boost_vals = np.linspace(
    min(hist_boost.min(), prop_boost.min()) - 0.6,
    max(hist_boost.max(), prop_boost.max()) + 0.6,
    220,
 )
M1, BP = np.meshgrid(mass1_vals, boost_vals)

# Historical surface via 2D histogram density (no external dependencies).
x_edges = np.linspace(0.0, 30.0, 55)
y_edges = np.linspace(boost_vals.min(), boost_vals.max(), 55)
hist2d, _, _ = np.histogram2d(hist_mass1, hist_boost, bins=[x_edges, y_edges])
hist2d = hist2d.T

# Lightweight smoothing by neighbor averaging for a cleaner surface.
kernel = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype=float)
kernel = kernel / kernel.sum()
padded = np.pad(hist2d, 1, mode='edge')
smooth = np.zeros_like(hist2d)
for i in range(hist2d.shape[0]):
    for j in range(hist2d.shape[1]):
        smooth[i, j] = np.sum(padded[i:i+3, j:j+3] * kernel)

# Map binned density to display grid.
x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])
density_surface = np.zeros_like(M1)
x_idx = np.clip(np.searchsorted(x_centers, M1) - 1, 0, len(x_centers) - 1)
y_idx = np.clip(np.searchsorted(y_centers, BP) - 1, 0, len(y_centers) - 1)
density_surface = smooth[y_idx, x_idx]

# Construct projected boundaries from stated constraints.
boost_lower = np.piecewise(
    mass1_vals,
    [
        (mass1_vals >= 0.0) & (mass1_vals < 10.0),
        (mass1_vals >= 10.0) & (mass1_vals < 20.0),
        (mass1_vals >= 20.0) & (mass1_vals <= 30.0),
    ],
    [
        lambda x: 0.0922 * (x + 0.5) + 0.8378 - 0.5,
        lambda x: 0.0922 * (x + 0.9) + 0.8378 - 0.5,
        lambda x: 0.0922 * (x + 0.0) + 0.8378 - 0.5,
    ],
)
boost_upper = np.piecewise(
    mass1_vals,
    [
        (mass1_vals >= 0.0) & (mass1_vals < 10.0),
        (mass1_vals >= 10.0) & (mass1_vals < 20.0),
        (mass1_vals >= 20.0) & (mass1_vals <= 30.0),
    ],
    [
        lambda x: 0.0922 * (x + 3.5) + 0.8378 + 0.5,
        lambda x: 0.0922 * (x + 3.0) + 0.8378 + 0.5,
        lambda x: 0.0922 * (x + 1.5) + 0.8378 + 0.5,
    ],
)

# Boundary-based feasibility surface.
clearance_surface = np.minimum(BP - boost_lower[None, :], boost_upper[None, :] - BP)
feasible_surface = np.maximum(clearance_surface, 0.0)

# Z for proposed points on both surfaces.
prop_density_z = density_surface[
    np.clip(np.searchsorted(boost_vals, prop_boost) - 1, 0, len(boost_vals) - 1),
    np.clip(np.searchsorted(mass1_vals, prop_mass1) - 1, 0, len(mass1_vals) - 1),
]
prop_lower = np.interp(prop_mass1, mass1_vals, boost_lower)
prop_upper = np.interp(prop_mass1, mass1_vals, boost_upper)
prop_clearance_z = np.maximum(np.minimum(prop_boost - prop_lower, prop_upper - prop_boost), 0.0)

fig = plt.figure(figsize=(22, 7))
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax2 = fig.add_subplot(1, 3, 2, projection='3d')
ax3 = fig.add_subplot(1, 3, 3, projection='3d')

# 1) Historical surface only
surf1 = ax1.plot_surface(M1, BP, density_surface, cmap='viridis', alpha=0.9, edgecolor='none')
ax1.set_title('Historical Surface Only (197 points)')
ax1.set_xlabel('Mass1 [kg/h]')
ax1.set_ylabel('Boost pressure [bar]')
ax1.set_zlabel('Point density')
ax1.view_init(elev=28, azim=-125)
fig.colorbar(surf1, ax=ax1, shrink=0.72, pad=0.04)

# 2) Historical surface + boundaries + proposed points
surf2 = ax2.plot_surface(M1, BP, density_surface, cmap='cividis', alpha=0.78, edgecolor='none')
ax2.plot(mass1_vals, boost_lower, np.zeros_like(mass1_vals), '--', color='crimson', lw=2.2, label='Lower boundary')
ax2.plot(mass1_vals, boost_upper, np.zeros_like(mass1_vals), '--', color='goldenrod', lw=2.2, label='Upper boundary')
ax2.plot([30.0, 30.0], [boost_vals.min(), boost_vals.max()], [0.0, 0.0], ':', color='black', lw=2.0, label='Mass1 limit')
ax2.plot(prop_mass1, prop_boost, prop_density_z, '-o', color='orangered', lw=2.6, markersize=6, label='Proposed points')
ax2.set_title('Historical + Boundaries + Proposed')
ax2.set_xlabel('Mass1 [kg/h]')
ax2.set_ylabel('Boost pressure [bar]')
ax2.set_zlabel('Point density')
ax2.view_init(elev=28, azim=-125)
ax2.legend(loc='best')
fig.colorbar(surf2, ax=ax2, shrink=0.72, pad=0.04)

# 3) Boundary surface + historical markers + proposed points
surf3 = ax3.plot_surface(M1, BP, feasible_surface, cmap='plasma', alpha=0.85, edgecolor='none')
ax3.scatter(hist_mass1, hist_boost, np.zeros_like(hist_mass1), c='royalblue', s=12, alpha=0.35, label='Historical markers')
ax3.plot(mass1_vals, boost_lower, np.zeros_like(mass1_vals), '--', color='crimson', lw=2.2, label='Lower boundary')
ax3.plot(mass1_vals, boost_upper, np.zeros_like(mass1_vals), '--', color='goldenrod', lw=2.2, label='Upper boundary')
ax3.plot([30.0, 30.0], [boost_vals.min(), boost_vals.max()], [0.0, 0.0], ':', color='black', lw=2.0, label='Mass1 limit')
ax3.plot(prop_mass1, prop_boost, prop_clearance_z, '-o', color='orangered', lw=2.6, markersize=6, label='Proposed points')
ax3.set_title('Boundary Surface + Historical Markers')
ax3.set_xlabel('Mass1 [kg/h]')
ax3.set_ylabel('Boost pressure [bar]')
ax3.set_zlabel('Feasibility clearance')
ax3.view_init(elev=28, azim=-125)
ax3.legend(loc='best')
fig.colorbar(surf3, ax=ax3, shrink=0.72, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# GP uncertainty surface in Mass1 vs Boost plane (no IEMP output surface)
# This uses total predictive variance across all outputs, with boundary guidance lines and historical points.

mass1_idx = input_features.index('Mass1')
boost_idx = input_features.index('Boost pressure')
mass2_idx = input_features.index('Mass2')

grid_n = 90
mass1_axis = np.linspace(0.0, 30.0, grid_n)
boost_axis = np.linspace(
    float(pd_df['Boost pressure'].min()) - 0.4,
    float(pd_df['Boost pressure'].max()) + 0.4,
    grid_n,
 )
M1, BP = np.meshgrid(mass1_axis, boost_axis)

base = current_loc.squeeze().clone()
query = base.repeat(grid_n * grid_n, 1)

m1_scaled = (
    torch.tensor(M1.reshape(-1), dtype=torch.float64)
    - torch.tensor(scaler_x.scaler.mean_[mass1_idx], dtype=torch.float64)
 ) / torch.tensor(scaler_x.scaler.scale_[mass1_idx], dtype=torch.float64)
bp_scaled = (
    torch.tensor(BP.reshape(-1), dtype=torch.float64)
    - torch.tensor(scaler_x.scaler.mean_[boost_idx], dtype=torch.float64)
 ) / torch.tensor(scaler_x.scaler.scale_[boost_idx], dtype=torch.float64)

query[:, mass1_idx] = m1_scaled
query[:, boost_idx] = bp_scaled

with torch.no_grad():
    var_parts = [
        model.posterior(query).variance.squeeze(-1).cpu().numpy()
        for model in agent.models
    ]
    gp_var_surface = np.sum(np.stack(var_parts, axis=0), axis=0).reshape(grid_n, grid_n)

mass2_fixed = float(scaler_x.inverse_transform(current_loc[0, mass2_idx].item(), mass2_idx))
line_x = np.linspace(0.0, 30.0, 300)
line_center = 0.0922 * (line_x + mass2_fixed) + 0.8378
line_low = line_center - 0.5
line_high = line_center + 0.5
eff_load_limit = min(30.0, 30.0 - mass2_fixed)

def _var_line_z(x_line, y_line):
    q = base.repeat(len(x_line), 1)
    x_scaled = (
        torch.tensor(x_line, dtype=torch.float64)
        - torch.tensor(scaler_x.scaler.mean_[mass1_idx], dtype=torch.float64)
    ) / torch.tensor(scaler_x.scaler.scale_[mass1_idx], dtype=torch.float64)
    y_scaled = (
        torch.tensor(y_line, dtype=torch.float64)
        - torch.tensor(scaler_x.scaler.mean_[boost_idx], dtype=torch.float64)
    ) / torch.tensor(scaler_x.scaler.scale_[boost_idx], dtype=torch.float64)
    q[:, mass1_idx] = x_scaled
    q[:, boost_idx] = y_scaled
    with torch.no_grad():
        vals = [
            model.posterior(q).variance.squeeze(-1).cpu().numpy()
            for model in agent.models
        ]
    return np.sum(np.stack(vals, axis=0), axis=0)

mask = (line_low >= boost_axis.min()) & (line_high <= boost_axis.max())
bx = line_x[mask]
bl = line_low[mask]
bh = line_high[mask]
z_low = _var_line_z(bx, bl)
z_high = _var_line_z(bx, bh)

# Historical and proposed points projected on the same uncertainty surface
hist_m1 = pd_df['Mass1'].to_numpy()
hist_bp = pd_df['Boost pressure'].to_numpy()
hist_z = _var_line_z(hist_m1, hist_bp)

prop_m1 = path_original['Mass1'].to_numpy()
prop_bp = path_original['Boost pressure'].to_numpy()
prop_z = _var_line_z(prop_m1, prop_bp)

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(M1, BP, gp_var_surface, cmap='plasma', alpha=0.86, edgecolor='none')
ax.scatter(hist_m1, hist_bp, hist_z, c='royalblue', s=14, alpha=0.35, label='Historical data')
ax.plot(bx, bl, z_low, '--', color='crimson', lw=2.4, label='Boost lower boundary')
ax.plot(bx, bh, z_high, '--', color='goldenrod', lw=2.4, label='Boost upper boundary')

if eff_load_limit >= 0.0 and eff_load_limit <= 30.0:
    yv = np.linspace(boost_axis.min(), boost_axis.max(), 120)
    xv = np.full_like(yv, eff_load_limit)
    zv = _var_line_z(xv, yv)
    ax.plot(xv, yv, zv, ':', color='black', lw=2.0, label='Load boundary')

ax.plot(prop_m1, prop_bp, prop_z, '-o', color='orangered', lw=2.8, markersize=6, label='Proposed points')
for i, (xv, yv, zv) in enumerate(zip(prop_m1, prop_bp, prop_z), start=1):
    ax.text(xv, yv, zv, f'S{i}', color='orangered', fontsize=9)

ax.set_xlabel('Mass1 [kg/h]')
ax.set_ylabel('Boost pressure [bar]')
ax.set_zlabel('Total predictive variance')
ax.set_title('GP Uncertainty Surface with Boundary Guidance and Historical Data')
ax.view_init(elev=28, azim=-125)
ax.legend(loc='best')
fig.colorbar(surf, ax=ax, shrink=0.72, pad=0.08, label='Total predictive variance')

plt.tight_layout()
plt.show()

In [ ]:
# GP predictive mean surface in Mass1 vs Boost plane with historical and suggested points
# z-axis uses the same scalar predictive-mean definition as the previous cell.

mass1_idx = input_features.index('Mass1')
boost_idx = input_features.index('Boost pressure')
mass2_idx = input_features.index('Mass2')

# Keep exactly the same scalarization to make both mean surfaces comparable.
z_mode = 'single_output'
output_index = 0
if output_index < 0 or output_index >= len(agent.models):
    raise ValueError(f'output_index must be in [0, {len(agent.models)-1}]')

grid_n = 90
mass1_axis = np.linspace(0.0, 30.0, grid_n)
boost_axis = np.linspace(
    float(pd_df['Boost pressure'].min()) - 0.4,
    float(pd_df['Boost pressure'].max()) + 0.4,
    grid_n,
 )
M1, BP = np.meshgrid(mass1_axis, boost_axis)

base = current_loc.squeeze().clone()
query = base.repeat(grid_n * grid_n, 1)

m1_scaled = (
    torch.tensor(M1.reshape(-1), dtype=torch.float64)
    - torch.tensor(scaler_x.scaler.mean_[mass1_idx], dtype=torch.float64)
) / torch.tensor(scaler_x.scaler.scale_[mass1_idx], dtype=torch.float64)
bp_scaled = (
    torch.tensor(BP.reshape(-1), dtype=torch.float64)
    - torch.tensor(scaler_x.scaler.mean_[boost_idx], dtype=torch.float64)
) / torch.tensor(scaler_x.scaler.scale_[boost_idx], dtype=torch.float64)

query[:, mass1_idx] = m1_scaled
query[:, boost_idx] = bp_scaled

def _scalar_predictive_mean(q):
    with torch.no_grad():
        all_means = np.stack(
            [mdl.posterior(q).mean.squeeze(-1).cpu().numpy() for mdl in agent.models],
            axis=0,
        )
    if z_mode == 'single_output':
        return all_means[output_index]
    if z_mode == 'mean_across_outputs':
        return np.mean(all_means, axis=0)
    raise ValueError("z_mode must be 'single_output' or 'mean_across_outputs'")

def _mean_line_z(x_line, y_line):
    q = base.repeat(len(x_line), 1)
    x_scaled = (
        torch.tensor(x_line, dtype=torch.float64)
        - torch.tensor(scaler_x.scaler.mean_[mass1_idx], dtype=torch.float64)
    ) / torch.tensor(scaler_x.scaler.scale_[mass1_idx], dtype=torch.float64)
    y_scaled = (
        torch.tensor(y_line, dtype=torch.float64)
        - torch.tensor(scaler_x.scaler.mean_[boost_idx], dtype=torch.float64)
    ) / torch.tensor(scaler_x.scaler.scale_[boost_idx], dtype=torch.float64)
    q[:, mass1_idx] = x_scaled
    q[:, boost_idx] = y_scaled
    return _scalar_predictive_mean(q)

gp_mean_surface = _scalar_predictive_mean(query).reshape(grid_n, grid_n)

# Guidance boundaries in Mass1-Boost plane at the current Mass2 slice
mass2_fixed = float(scaler_x.inverse_transform(current_loc[0, mass2_idx].item(), mass2_idx))
line_x = np.linspace(0.0, 30.0, 300)
line_center = 0.0922 * (line_x + mass2_fixed) + 0.8378
line_low = line_center - 0.5
line_high = line_center + 0.5
eff_load_limit = min(30.0, 30.0 - mass2_fixed)

mask = (line_low >= boost_axis.min()) & (line_high <= boost_axis.max())
bx = line_x[mask]
bl = line_low[mask]
bh = line_high[mask]
z_low = _mean_line_z(bx, bl)
z_high = _mean_line_z(bx, bh)

# Historical and suggested points projected on GP mean surface
hist_m1 = pd_df['Mass1'].to_numpy()
hist_bp = pd_df['Boost pressure'].to_numpy()
hist_z = _mean_line_z(hist_m1, hist_bp)

prop_m1 = path_original['Mass1'].to_numpy()
prop_bp = path_original['Boost pressure'].to_numpy()
prop_z = _mean_line_z(prop_m1, prop_bp)

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(M1, BP, gp_mean_surface, cmap='viridis', alpha=0.86, edgecolor='none')
ax.scatter(hist_m1, hist_bp, hist_z, c='royalblue', s=14, alpha=0.35, label='Historical data')
ax.plot(bx, bl, z_low, '--', color='crimson', lw=2.4, label='Boost lower boundary')
ax.plot(bx, bh, z_high, '--', color='goldenrod', lw=2.4, label='Boost upper boundary')

if eff_load_limit >= 0.0 and eff_load_limit <= 30.0:
    yv = np.linspace(boost_axis.min(), boost_axis.max(), 120)
    xv = np.full_like(yv, eff_load_limit)
    zv = _mean_line_z(xv, yv)
    ax.plot(xv, yv, zv, ':', color='black', lw=2.0, label='Load boundary')

ax.plot(prop_m1, prop_bp, prop_z, '-o', color='orangered', lw=2.8, markersize=6, label='Suggested points')
for i, (xv, yv, zv) in enumerate(zip(prop_m1, prop_bp, prop_z), start=1):
    ax.text(xv, yv, zv, f'S{i}', color='orangered', fontsize=9)

ax.set_xlabel('Mass1 [kg/h]')
ax.set_ylabel('Boost pressure [bar]')
ax.set_zlabel('Predictive mean')
ax.set_title('GP Predictive Mean Surface with Boundary Guidance')
ax.set_zlim(float(gp_mean_surface.min()), float(gp_mean_surface.max()))
ax.view_init(elev=28, azim=-125)
ax.legend(loc='best')
fig.colorbar(surf, ax=ax, shrink=0.72, pad=0.08, label='Predictive mean')

plt.tight_layout()
plt.show()

In [ ]:
# Predictive mean slice for Mass1 and Boost pressure only
# X: Mass1 [kg/h], Y: Boost pressure [bar], Z: predictive mean.
# Other input dimensions remain fixed at the current operating point.

mass1_idx = input_features.index('Mass1')
boost_idx = input_features.index('Boost pressure')

grid_n = 100
mass1_axis = np.linspace(float(pd_df['Mass1'].min()), float(pd_df['Mass1'].max()), grid_n)
boost_axis = np.linspace(float(pd_df['Boost pressure'].min()), float(pd_df['Boost pressure'].max()), grid_n)
M1, BP = np.meshgrid(mass1_axis, boost_axis)

base = current_loc.squeeze().clone()
query = base.repeat(grid_n * grid_n, 1)

m1_scaled = (
    torch.tensor(M1.reshape(-1), dtype=torch.float64)
    - torch.tensor(scaler_x.scaler.mean_[mass1_idx], dtype=torch.float64)
) / torch.tensor(scaler_x.scaler.scale_[mass1_idx], dtype=torch.float64)
bp_scaled = (
    torch.tensor(BP.reshape(-1), dtype=torch.float64)
    - torch.tensor(scaler_x.scaler.mean_[boost_idx], dtype=torch.float64)
) / torch.tensor(scaler_x.scaler.scale_[boost_idx], dtype=torch.float64)

query[:, mass1_idx] = m1_scaled
query[:, boost_idx] = bp_scaled

with torch.no_grad():
    mean_stack = np.stack(
        [mdl.posterior(query).mean.squeeze(-1).cpu().numpy() for mdl in agent.models],
        axis=0,
    )
Z = np.mean(mean_stack, axis=0).reshape(grid_n, grid_n)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(M1, BP, Z, cmap='viridis', alpha=0.9, edgecolor='none')

ax.set_xlabel('Mass1 [kg/h]')
ax.set_ylabel('Boost pressure [bar]')
ax.set_zlabel('Predictive mean')
ax.set_title('Predictive Mean Slice for Mass1 and Boost Pressure')
ax.view_init(elev=28, azim=-125)
fig.colorbar(surf, ax=ax, shrink=0.72, pad=0.08, label='Predictive mean')

plt.tight_layout()
plt.show()

In [ ]:
# Global RMSE reduction curve from proposed points (expected RMSE proxy)
# This computes expected RMSE by mapping predictive variance -> RMSE on historical inputs.
# It then conditions each GP with proposed points (fantasy mean observations) step by step.

eval_x = torch.tensor(inputs_scaled, dtype=torch.float64)
proposed_x = path_scaled if isinstance(path_scaled, torch.Tensor) else torch.tensor(path_scaled, dtype=torch.float64)
proposed_x = proposed_x.to(dtype=torch.float64)

def _global_expected_rmse(models, x_eval, scaler_y):
    rmse_sq_per_output = []
    for j, mdl in enumerate(models):
        mdl.eval()
        with torch.no_grad():
            var_scaled = mdl.posterior(x_eval).variance.squeeze(-1).cpu().numpy()
        mse_scaled = float(np.mean(var_scaled))
        # Convert scaled-output RMSE to original units using output scaler scale.
        rmse_orig = np.sqrt(max(mse_scaled, 0.0)) * float(scaler_y.scaler.scale_[j])
        rmse_sq_per_output.append(rmse_orig ** 2)
    return float(np.sqrt(np.mean(rmse_sq_per_output)))

rmse_curve = []
step_labels = ['Baseline']

# Baseline expected RMSE
rmse_curve.append(_global_expected_rmse(agent.models, eval_x, scaler_y))

# Sequentially add proposed points and recompute expected RMSE
for k in range(1, proposed_x.shape[0] + 1):
    xk = proposed_x[:k]
    fantasy_models = []
    for mdl in agent.models:
        mdl.eval()
        with torch.no_grad():
            yk = mdl.posterior(xk).mean
        fantasy_models.append(mdl.condition_on_observations(X=xk, Y=yk))

    rmse_curve.append(_global_expected_rmse(fantasy_models, eval_x, scaler_y))
    step_labels.append(f'After {k} point' if k == 1 else f'After {k} points')

rmse_curve = np.array(rmse_curve, dtype=float)
rmse_drop = rmse_curve[0] - rmse_curve
rmse_drop_pct = 100.0 * rmse_drop / max(rmse_curve[0], 1e-12)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(step_labels, rmse_curve, marker='o', linewidth=2.2, color='navy', label='Global expected RMSE')
ax1.set_ylabel('Global expected RMSE (original output units)')
ax1.set_title('Global RMSE Curve from Proposed Points')
ax1.grid(axis='y', alpha=0.25)

ax2 = ax1.twinx()
ax2.bar(step_labels, rmse_drop_pct, alpha=0.22, color='seagreen', label='RMSE reduction [%]')
ax2.set_ylabel('Reduction [% from baseline]')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

plt.tight_layout()
plt.show()

pd.DataFrame({
    'step': step_labels,
    'global_expected_rmse': rmse_curve,
    'rmse_reduction': rmse_drop,
    'rmse_reduction_percent': rmse_drop_pct,
})

In [ ]:
# Expected variance reduction from proposed points (no measured outputs required)
# Uses GP fantasy conditioning and evaluates integrated variance over a domain-wide Sobol set.

# Why this matters: evaluating only on training inputs can look flat because those points are already
# highly certain under the fitted GP.

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.quasirandom import SobolEngine

proposed_x = path_scaled if isinstance(path_scaled, torch.Tensor) else torch.tensor(path_scaled, dtype=torch.float64)
proposed_x = proposed_x.to(dtype=torch.float64)

# Evaluation mode: 'sobol_domain' (recommended) or 'train_inputs'
eval_mode = 'sobol_domain'
n_eval = 3000

if eval_mode == 'sobol_domain':
    sobol = SobolEngine(dimension=agent.d, scramble=True)
    eval_x = agent.lower + (agent.upper - agent.lower) * sobol.draw(n_eval)
elif eval_mode == 'train_inputs':
    eval_x = torch.tensor(inputs_scaled, dtype=torch.float64)
else:
    raise ValueError("eval_mode must be 'sobol_domain' or 'train_inputs'")

# Optional side-by-side check to expose saturation on training points
also_report_train_grid = True
eval_x_train = torch.tensor(inputs_scaled, dtype=torch.float64)

def _integrated_variance_by_output(models, x_eval):
    vals = []
    for mdl in models:
        mdl.eval()
        with torch.no_grad():
            v = mdl.posterior(x_eval).variance.squeeze(-1).cpu().numpy()
        vals.append(float(np.mean(v)))
    return np.array(vals, dtype=float)

def _global_integrated_variance(per_output_iv):
    return float(np.sqrt(np.mean(per_output_iv ** 2)))

def _iv_matrix_over_steps(base_models, x_eval, x_steps):
    rows = []
    rows.append(_integrated_variance_by_output(base_models, x_eval))

    for k in range(1, x_steps.shape[0] + 1):
        xk = x_steps[:k]
        fantasy_models = []
        for mdl in base_models:
            mdl.eval()
            with torch.no_grad():
                yk = mdl.posterior(xk).mean
            fantasy_models.append(mdl.condition_on_observations(X=xk, Y=yk))
        rows.append(_integrated_variance_by_output(fantasy_models, x_eval))

    return np.vstack(rows)

step_labels = ['Baseline'] + [f'After {k} point' if k == 1 else f'After {k} points' for k in range(1, proposed_x.shape[0] + 1)]
iv_matrix = _iv_matrix_over_steps(agent.models, eval_x, proposed_x)
global_iv_curve = np.array([_global_integrated_variance(row) for row in iv_matrix])

baseline_global_iv = global_iv_curve[0]
global_iv_drop = baseline_global_iv - global_iv_curve
global_iv_drop_pct = 100.0 * global_iv_drop / max(baseline_global_iv, 1e-12)

# Per-output reduction table (baseline vs final)
baseline_per_output = iv_matrix[0]
final_per_output = iv_matrix[-1]
per_output_report = pd.DataFrame({
    'output': output_features,
    'iv_baseline': baseline_per_output,
    'iv_final': final_per_output,
    'iv_drop': baseline_per_output - final_per_output,
    'iv_drop_percent': 100.0 * (baseline_per_output - final_per_output) / np.maximum(baseline_per_output, 1e-12),
})

# Optional train-grid comparison curve
train_curve = None
if also_report_train_grid:
    iv_train = _iv_matrix_over_steps(agent.models, eval_x_train, proposed_x)
    train_curve = np.array([_global_integrated_variance(row) for row in iv_train])
    train_drop_pct = 100.0 * (train_curve[0] - train_curve[-1]) / max(train_curve[0], 1e-12)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(step_labels, global_iv_curve, marker='o', linewidth=2.2, color='navy', label=f'Global IV ({eval_mode})')
if train_curve is not None:
    ax1.plot(step_labels, train_curve, marker='o', linewidth=1.6, color='gray', alpha=0.7, linestyle='--', label='Global IV (train_inputs)')
ax1.set_ylabel('Global integrated variance (scaled-output units)')
ax1.set_title('Expected Variance Reduction from Proposed Points')
ax1.grid(axis='y', alpha=0.25)

ax2 = ax1.twinx()
ax2.bar(step_labels, global_iv_drop_pct, alpha=0.22, color='seagreen', label='Variance reduction [%]')
ax2.set_ylabel('Reduction [% from baseline]')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

plt.tight_layout()
plt.show()

print(f'Eval mode            : {eval_mode}')
print(f'Baseline global IV   : {baseline_global_iv:.6e}')
print(f'Final global IV      : {global_iv_curve[-1]:.6e}')
print(f'Global IV reduction  : {global_iv_drop[-1]:.6e} ({global_iv_drop_pct[-1]:.2f}%)')
if train_curve is not None:
    print(f'Train-grid reduction : {train_drop_pct:.6f}%')

display(pd.DataFrame({
    'step': step_labels,
    'global_integrated_variance': global_iv_curve,
    'global_iv_reduction': global_iv_drop,
    'global_iv_reduction_percent': global_iv_drop_pct,
}))

display(per_output_report.sort_values('iv_drop', ascending=False).reset_index(drop=True))

In [ ]:
# EVR paralysis diagnostics: training-grid saturation vs path novelty
from torch.quasirandom import SobolEngine

# 1) How much does the planned path move?
step_move_scaled = torch.norm(path_scaled[1:] - path_scaled[:-1], dim=1).cpu().numpy() if path_scaled.shape[0] > 1 else np.array([])
print('Step movement in scaled space:', step_move_scaled)

# 2) How close are proposed points to existing training data?
nn_dist_scaled = torch.cdist(path_scaled, agent.X).min(dim=1).values.cpu().numpy()
print('Nearest-train distance per proposed point (scaled):', nn_dist_scaled)
print('Mean nearest-train distance:', float(nn_dist_scaled.mean()))

# 3) Compare EVR measured on training inputs vs random domain inputs
eval_train = torch.tensor(inputs_scaled, dtype=torch.float64)
sobol = SobolEngine(dimension=agent.d, scramble=True)
eval_rand = agent.lower + (agent.upper - agent.lower) * sobol.draw(3000)

def iv_curve_for_eval(eval_x):
    iv_rows = []
    base = np.array([
        float(np.mean(m.posterior(eval_x).variance.squeeze(-1).detach().cpu().numpy()))
        for m in agent.models
    ])
    iv_rows.append(base)

    for k in range(1, path_scaled.shape[0] + 1):
        xk = path_scaled[:k]
        fmods = []
        for m in agent.models:
            with torch.no_grad():
                yk = m.posterior(xk).mean
            fmods.append(m.condition_on_observations(X=xk, Y=yk))
        row = np.array([
            float(np.mean(m.posterior(eval_x).variance.squeeze(-1).detach().cpu().numpy()))
            for m in fmods
        ])
        iv_rows.append(row)

    iv_rows = np.vstack(iv_rows)
    global_iv = np.sqrt(np.mean(iv_rows**2, axis=1))
    return global_iv

g_train = iv_curve_for_eval(eval_train)
g_rand = iv_curve_for_eval(eval_rand)

red_train = (g_train[0] - g_train[-1]) / max(g_train[0], 1e-12) * 100.0
red_rand = (g_rand[0] - g_rand[-1]) / max(g_rand[0], 1e-12) * 100.0

print(f'Train-grid EVR: {red_train:.6f}%')
print(f'Random-domain EVR: {red_rand:.6f}%')
print('Train global IV curve:', g_train)
print('Random global IV curve:', g_rand)

## New modified planner

In [ ]:
from src.soed.agents.constrained_multistep_mimo_agent_service import ConstrainedMultiStepMIMOAgentService

agent_service = ConstrainedMultiStepMIMOAgentService(
    bounds=scaled_bounds,
    feature_names=input_features,
    scaler_x=scaler_x,
    mass1_name="Mass1",
    mass2_name="Mass2",
    boost_name="Boost pressure",
    load_limit=30.0,
    boost_slope=0.0922,
    boost_intercept=0.8378,
    boost_band=0.5,
    min_load=3.0,
    ambient_pressure=1.0,
    tc_boost_limit=3.8,
)

agent_service.fit_data(inputs_scaled, outputs_scaled)
path_generated_by_agent_service = agent_service.plan_multistep_batch(
    current_location=current_loc,
    q_steps=3,
    num_scenarios=600,
    w_dist=0.01,
    enforce_feasible_sampling=True,
    feasible_margin_weight=1.0,
)

In [ ]:
path_selected_original = agent_service._to_original_units(path_generated_by_agent_service)
path_mask = agent_service.constraints.feasible_mask(path_selected_original)
print("selected path feasible:", bool(path_mask.all()))
print(path_selected_original)

In [ ]:
sobol = agent_service.planner_service.last_sobol_pool
sobol_original = agent_service._to_original_units(sobol)
mask = agent_service.constraints.feasible_mask(sobol_original)
print("sobol count:", sobol.shape[0])
print("feasible count:", int(mask.sum()))
print("outside count:", int((~mask).sum()))
print("feasibility checked in original engineering units after inverse transform")

In [ ]:
# Convert recommendation to physical units and verify all constraints
path_original = []
for p in path_generated_by_agent_service:
    row = []
    for i in range(len(input_features)):
        row.append(scaler_x.inverse_transform(p[i].item(), i))
    path_original.append(row)

path_original = pd.DataFrame(path_original, columns=input_features)

mass_sum = path_original['Mass1'] + path_original['Mass2']
boost_center = 0.0922 * mass_sum + 0.8378

# Pull hard bounds from agent so validation mirrors planner logic.
min_load = float(getattr(agent_service, 'min_load', 3.0))
max_load = float(getattr(agent_service, 'load_limit', 30.0))
ambient_pressure = float(getattr(agent_service, 'ambient_pressure', 1.0))
tc_boost_limit = float(getattr(agent_service, 'tc_boost_limit', 3.8))

path_original['Mass1+Mass2'] = mass_sum
path_original['Boost_low_raw'] = boost_center - 0.5
path_original['Boost_high_raw'] = boost_center + 0.5
path_original['Boost_low'] = np.maximum(path_original['Boost_low_raw'], ambient_pressure)
path_original['Boost_high'] = np.minimum(path_original['Boost_high_raw'], tc_boost_limit)

# Load check aligned with planner constraints.
path_original['Load_ok'] = (
    (path_original['Mass1'] >= 0.0) &
    (path_original['Mass1'] < max_load) &
    (mass_sum >= min_load) &
    (mass_sum < max_load)
)

# Boost check uses slanted band clipped by floor/roof, aligned with planner.
path_original['Boost_ok'] = (
    (path_original['Boost pressure'] >= path_original['Boost_low']) &
    (path_original['Boost pressure'] <= path_original['Boost_high'])
)

# BR limit by Mass1 band (aligned with planner band edges).
path_original['BR_ok'] = (
    ((path_original['Mass1'] >= 0.0) & (path_original['Mass1'] < 10.0) & (path_original['Mass2'] > 0.5) & (path_original['Mass2'] < 3.5)) |
    ((path_original['Mass1'] >= 10.0) & (path_original['Mass1'] < 20.0) & (path_original['Mass2'] > 0.9) & (path_original['Mass2'] < 3.0)) |
    ((path_original['Mass1'] >= 20.0) & (path_original['Mass1'] < 30.0) & (path_original['Mass2'] > 0.0) & (path_original['Mass2'] < 1.5))
)

# VVA limit by Mass1 band
low_load = (path_original['Mass1'] >= 0.0) & (path_original['Mass1'] < 10.0)
mid_load = (path_original['Mass1'] >= 10.0) & (path_original['Mass1'] < 20.0)
high_load = (path_original['Mass1'] >= 20.0) & (path_original['Mass1'] < 30.0)

path_original['VVA_ok'] = (
    (
        low_load &
        (path_original['IVO'] >= 350.0) & (path_original['IVO'] <= 435.0) &
        (path_original['IVC'] >= 500.0) & (path_original['IVC'] <= 540.0) &
        (path_original['EVO'] >= 128.0) & (path_original['EVO'] <= 218.0) &
        (path_original['EVC'] >= 270.0) & (path_original['EVC'] <= 350.0)
    ) |
    (
        mid_load &
        (path_original['IVO'] >= 330.0) & (path_original['IVO'] <= 390.0) &
        (path_original['IVC'] >= 500.0) & (path_original['IVC'] <= 570.0) &
        (path_original['EVO'] >= 128.0) & (path_original['EVO'] <= 218.0) &
        (path_original['EVC'] >= 330.0) & (path_original['EVC'] <= 370.0)
    ) |
    (
        high_load &
        (path_original['IVO'] >= 345.0) & (path_original['IVO'] <= 365.0) &
        (path_original['IVC'] >= 495.0) & (path_original['IVC'] <= 535.0) &
        (path_original['EVO'] >= 128.0) & (path_original['EVO'] <= 218.0) &
        (path_original['EVC'] >= 345.0) & (path_original['EVC'] <= 355.0)
    )
)

path_original['Feasible'] = path_original['Load_ok'] & path_original['Boost_ok'] & path_original['BR_ok'] & path_original['VVA_ok']

# Compact display columns for decision making.
display_cols = [
    'Engine_speed', 'Boost pressure', 'Mass1', 'Mass2', 'SOI2', 'IVO', 'IVC', 'EVO', 'EVC',
    'Mass1+Mass2', 'Boost_low', 'Boost_high', 'Load_ok', 'Boost_ok', 'BR_ok', 'VVA_ok', 'Feasible'
]
path_original[display_cols]

In [ ]:
# Final cell: 2D engineering constraint view with the active planner path
hist = pd_df.copy()
hist['Mass1+Mass2'] = hist['Mass1'] + hist['Mass2']

x = np.linspace(hist['Mass1+Mass2'].min(), max(40.0, hist['Mass1+Mass2'].max()), 400)
y_mid = 0.0922 * x + 0.8378
y_low = y_mid - 0.5
y_high = y_mid + 0.5

min_load = float(getattr(agent_service, 'min_load', 3.0))
max_load = float(getattr(agent_service, 'load_limit', 30.0))
ambient_pressure = float(getattr(agent_service, 'ambient_pressure', 1.0))
tc_boost_limit = float(getattr(agent_service, 'tc_boost_limit', 3.8))

poly_low = np.maximum(y_low, ambient_pressure)
poly_high = np.minimum(y_high, tc_boost_limit)
poly_mask = (x >= min_load) & (x <= max_load) & (poly_high >= poly_low)

# Use the exact candidate pool generated during the planner run, not a new one.
candidate_pool = getattr(agent_service.planner_service, 'last_candidate_pool', None)
sobol_pool = getattr(agent_service.planner_service, 'last_sobol_pool', None)
local_pool = getattr(agent_service.planner_service, 'last_local_pool', None)

if candidate_pool is None or sobol_pool is None:
    from src.soed.services.planner_service import PlanningConfig
    planning_cfg = PlanningConfig(
        q_steps=3,
        num_scenarios=256,
        w_dist=1.0,
        enforce_feasible_sampling=False,
        include_local_paths=True,
        local_path_fraction=0.10,
        feasible_margin_weight=25.0,
    )
    candidate_pool, sobol_pool = agent_service.planner_service.candidate_generation.generate_mixed_paths(
        current_location=current_loc,
        q_steps=planning_cfg.q_steps,
        num_scenarios=planning_cfg.num_scenarios,
        config=planning_cfg,
    )
    local_pool = candidate_pool[sobol_pool.shape[0]:] if candidate_pool.shape[0] > sobol_pool.shape[0] else None

def _scaled_to_original_points(points):
    arr = points.detach().cpu().numpy() if hasattr(points, 'detach') else np.asarray(points, dtype=float)
    if arr.ndim == 3:
        arr = arr.reshape(-1, arr.shape[-1])
    df = pd.DataFrame(arr, columns=input_features)
    for col_name in input_features:
        col_idx = input_features.index(col_name)
        df[col_name] = scaler_x.inverse_transform(df[col_name].to_numpy(), col_idx)
    return df

sobol_df = _scaled_to_original_points(sobol_pool)
local_df = _scaled_to_original_points(local_pool) if local_pool is not None else pd.DataFrame(columns=input_features)
rec_df = path_original.copy()
if 'Mass1+Mass2' not in rec_df.columns:
    rec_df['Mass1+Mass2'] = rec_df['Mass1'] + rec_df['Mass2']
sobol_df['Mass1+Mass2'] = sobol_df['Mass1'] + sobol_df['Mass2']
local_df['Mass1+Mass2'] = local_df['Mass1'] + local_df['Mass2']

plt.figure(figsize=(10, 6))
plt.scatter(hist['Mass1+Mass2'], hist['Boost pressure'], s=25, c='#3b6fb6', alpha=0.65, label='Historical cases')
plt.scatter(sobol_df['Mass1+Mass2'], sobol_df['Boost pressure'], s=12, c='tab:blue', alpha=0.18, label='Sobol points')
plt.scatter(local_df['Mass1+Mass2'], local_df['Boost pressure'], s=18, c='tab:orange', alpha=0.48, label='Local forward points')
plt.plot(x, y_mid, '--', c='gray', lw=1.5, label='Boost center line')
plt.plot(x, y_low, '--', c='red', lw=1.5, label='Boost lower slanted limit')
plt.plot(x, y_high, '--', c='red', lw=1.5, label='Boost upper slanted limit')
plt.axhline(ambient_pressure, color='purple', linestyle='-.', lw=1.4, label='Ambient pressure floor')
plt.axhline(tc_boost_limit, color='brown', linestyle='-.', lw=1.4, label='TC boost roof')
plt.axvline(min_load, color='black', linestyle='-', lw=1.4, label='Min load wall')
plt.axvline(max_load, color='black', linestyle='-', lw=1.4, label='Max load wall')
plt.fill_between(
    x[poly_mask],
    poly_low[poly_mask],
    poly_high[poly_mask],
    color='limegreen',
    alpha=0.18,
    label='Feasible polygon region',
)

rec_x = rec_df['Mass1+Mass2'].to_numpy()
rec_y = rec_df['Boost pressure'].to_numpy()
plt.plot(rec_x, rec_y, '-o', c='orangered', lw=2.6, label='Recommended path')

for i, (xx, yy) in enumerate(zip(rec_x, rec_y), start=1):
    plt.text(xx + 0.25, yy + 0.02, f'Step {i}', color='orangered')

plt.xlabel('Mass1 + Mass2 [kg/h]')
plt.ylabel('Boost pressure [bar]')
plt.title('Constrained Planning Polygon and Candidate Exploration Points')
plt.grid(alpha=0.25)
plt.legend(loc='best')
plt.tight_layout()
plt.show()